# Budget Auction — A_distilled Experiment (Exp 1)

**Goal.** Test whether Variant A's architecture *can represent* Variant B's optimal mechanism,
or whether the A/B revenue gap is a true architectural capacity gap.

**Protocol (three stages):**
1. **Distillation** — Roll out trained B with truthful play; train a fresh A network to imitate
   B's outputs (alloc_probs, sigmoid_z) via supervised MSE. → `A_distilled` params.
2. **Direct evaluation** — Evaluate A_distilled as a mechanism (revenue, IC regret) *without*
   any IC fine-tuning. Tests how faithfully A's class can represent B's function.
3. **Warm-start fine-tuning** — Run IC-constrained nested-GD from A_distilled params (not
   random init). Tests whether A needs a better starting point, not a larger class.

**Interpretation:**

| Outcome | Conclusion |
|---|---|
| A_distilled revenue ≈ B | A's class CAN represent B's optimum; cold-start IC training fails to find it. Direct evidence for representational optimization gap. |
| A_distilled < B, warm-start closes gap | Same conclusion — A contains the solution but needs a better starting point. |
| Neither works | Possible true capacity gap; investigate further. |

**Prerequisite:** You must have a trained Variant B mechanism from Exp 1 binary search.
Load it by setting `B_MECH_PKL` below.

---
## Setup

In [ ]:
RUN_ENV = 'colab'   # ← change to 'local' for a local GPU run
LOCAL_PROJECT_DIR = None
# LOCAL_PROJECT_DIR = r'E:\Research\code\Dynamic Mechanism Design\Budget Auction'

In [ ]:
import os, sys, subprocess

if RUN_ENV == 'colab':
    subprocess.check_call(
        [sys.executable, '-m', 'pip', 'uninstall', '-y',
         'jax', 'jaxlib', 'jax-cuda12-plugin', 'jax-cuda12-pjrt'],
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    subprocess.check_call(
        [sys.executable, '-m', 'pip', 'install', '-qU',
         'jax[cuda12]', 'dm-haiku', 'optax', 'flax'])
    print('Install done. IMPORTANT: Runtime → Restart session, then re-run from this cell.')
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        os.chdir('/content/drive/MyDrive/My Research/Dynamic Mechanism Design/Budget Auction')
        print(f'Working directory: {os.getcwd()}')
    except ImportError:
        print('(Drive mount skipped — restart session first.)')
elif RUN_ENV == 'local':
    if LOCAL_PROJECT_DIR:
        os.chdir(LOCAL_PROJECT_DIR)
    print(f'Working directory: {os.getcwd()}')
else:
    raise ValueError(f'Unknown RUN_ENV: {RUN_ENV!r}')

In [ ]:
import jax
import jax.numpy as jnp
import haiku as hk
import optax
import matplotlib.pyplot as plt
import pickle, json, os, time
import numpy as np

from budget_auction.core import (
    BudgetSimulator, BoundedContPolicy,
    build_bounded_policy_net,
)
from budget_auction.mechanisms.budget_mechanism import BudgetMechanism
from budget_auction.mechanisms.input_adapters import VARIANT_NETWORK
from budget_auction.envs.public_budget_env import create_public_budget_auction
from budget_auction.train.distill import run_distillation
from budget_auction.train.run_acceptance import nested_GD_budget
from budget_auction.train.evaluate import eval_budget_mechanism_multikey, print_eval_report
from budget_auction.train.binary_search import (
    DEFAULT_BS_CFG, reopt_cumulative_ic,
    train_with_checkpoints, evaluate_checkpoint_list, run_binary_search,
)

print('JAX devices:', jax.devices())

---
## Configuration

Set `B_MECH_PKL` to the path of the trained Variant B mechanism pkl produced by the
Exp 1 binary search (`outputs/public_budget/binary_search/variant_B/best_mech_pg<X>.pkl`).

In [ ]:
# ── User-set variables ─────────────────────────────────────────────────────
SEED = 42

# Path to trained Variant B pkl from Exp 1 binary search.
# Find the exact filename in outputs/public_budget/binary_search/variant_B/
B_MECH_PKL = 'outputs/public_budget/binary_search/variant_B/best_mech_pg<X.XXX>.pkl'

# Output directory for distillation artifacts
DISTILL_SAVE_DIR = 'outputs/public_budget/distillation'

# Output directory for warm-start fine-tuning binary search
WARMSTART_SAVE_DIR = 'outputs/public_budget/binary_search/variant_A_distilled'

# ── Distillation hyperparameters ───────────────────────────────────────────
NUM_TRAJECTORIES = 2048   # rollouts of B; dataset size = NUM_TRAJECTORIES * T
DISTILL_EPOCHS   = 500
DISTILL_LR       = 1e-3
DISTILL_BATCH    = 256

# ── IC re-optimisation hyperparameters (for direct eval of A_distilled) ───
REOPT_STEPS    = 1000
REOPT_RESTARTS = 3
REOPT_LR       = 3e-3
REOPT_SAMPLES  = 128

# ── Warm-start fine-tuning (binary search) ─────────────────────────────────
# Use B's confirmed pg_lo as the starting upper bound
WARMSTART_BS_CFG = {
    **DEFAULT_BS_CFG,
    'lr_theta':        3e-4,
    'lr_w':            1e-3,
    'alpha_base':      1e5,
    'beta_weight':     500.0,
    'num_inner_steps': 300,
    'num_outer_epochs': 2000,
    'checkpoint_thresh':    0.15,
    'checkpoint_interval':  50,
    'pg_lo':   0.8,
    'pg_hi':   2.5,   # ← set to B's pg_lo or slightly above after distill eval
    'pg_base': 0.8,
    'pg_tol':  0.1,
    'ic_thresh':  0.05,
    'gap_thresh': 0.1,
    'ic_reopt_steps':    1000,
    'ic_reopt_restarts': 3,
    'ic_reopt_lr':       3e-3,
    'ic_reopt_samples':  128,
    'eval_num_keys':    32,
    'eval_num_samples': 256,
}

TIME_BUDGET_HOURS = 11.5

os.makedirs(DISTILL_SAVE_DIR, exist_ok=True)
os.makedirs(WARMSTART_SAVE_DIR, exist_ok=True)
print('Configuration set.')
print(f'  B pkl: {B_MECH_PKL}')
print(f'  Distill dir: {DISTILL_SAVE_DIR}')
print(f'  Warm-start dir: {WARMSTART_SAVE_DIR}')

---
## Build Auction and Networks

In [ ]:
auction = create_public_budget_auction(
    n_buyers=2, n_goods=1, horizon=5, discount=0.9,
    initial_budget=3.0, seed=SEED,
)
print(f'Auction: {auction.num_buyers}B × {auction.num_goods}G × T={auction.num_horizons}')
print(f'Initial budgets: {auction.initial_budgets}')

mechanism_net_A = hk.without_apply_rng(hk.transform(VARIANT_NETWORK['A']))
mechanism_net_B = hk.without_apply_rng(hk.transform(VARIANT_NETWORK['B']))
policy_network  = build_bounded_policy_net(
    d_priv=auction.d_priv, d_pub=auction.d_pub,
    G=auction.num_goods, hidden=128, residual_init_std=1.0,
)
print('Networks built: mechanism_net_A, mechanism_net_B, policy_network')

---
## Load Trained Variant B

Load the best Variant B mechanism from the Exp 1 binary search.
Check `outputs/public_budget/binary_search/variant_B/bs_summary.json` for
the `best_mechanism.mech_pkl` path if you don't remember the filename.

In [ ]:
# Auto-find best B pkl from bs_summary if B_MECH_PKL is still a placeholder
_b_pkl = B_MECH_PKL
if '<X.XXX>' in _b_pkl:
    _b_summary_path = 'outputs/public_budget/binary_search/variant_B/bs_summary.json'
    if os.path.exists(_b_summary_path):
        with open(_b_summary_path) as _f:
            _b_sum = json.load(_f)
        _b_pkl = _b_sum['best_mechanism']['mech_pkl']
        print(f'Auto-found B pkl: {_b_pkl}')
    else:
        raise FileNotFoundError(
            f'B_MECH_PKL is a placeholder and {_b_summary_path} does not exist.\n'
            'Set B_MECH_PKL manually in the Configuration cell.'
        )

with open(_b_pkl, 'rb') as _f:
    _b_ckpt = pickle.load(_f)

mechanism_B_params = _b_ckpt['mechanism_params']
mechanism_B = BudgetMechanism(mechanism_net_B, mechanism_B_params)

print(f'Loaded Variant B from: {_b_pkl}')
print(f'  profit_goal: {_b_ckpt["profit_goal"]:.3f}')
print(f'  epoch:       {_b_ckpt["epoch"]}')
print(f'  revenue_eval: {_b_ckpt["revenue_eval"]:.4f}')
print(f'  regret_eval:  {_b_ckpt["regret_eval"]:.4f}')

B_REVENUE_EVAL = _b_ckpt['revenue_eval']  # used as reference in final comparison

---
## Stage 1 — Distillation

Roll out B with truthful play to collect a supervised dataset, then train A
to minimise MSE between its outputs and B's.

**Dataset size:** `NUM_TRAJECTORIES × T = 2048 × 5 = 10,240` samples.

**Loss:** `MSE(alloc_A, alloc_B) + MSE(sigmoid(z_A), sigmoid(z_B))` — both in [0,1],
so the two terms are on equal footing at default weights (1.0, 1.0).

Expected runtime: ~10–20 min on GPU for 500 epochs over 10k samples.

In [ ]:
_t0 = time.time()

distilled_params, loss_history = run_distillation(
    auction          = auction,
    mechanism_net_A  = mechanism_net_A,
    mechanism_B      = mechanism_B,
    seed             = SEED,
    num_trajectories = NUM_TRAJECTORIES,
    num_epochs       = DISTILL_EPOCHS,
    lr               = DISTILL_LR,
    batch_size       = DISTILL_BATCH,
    alloc_weight     = 1.0,
    payment_weight   = 1.0,
    print_every      = 50,
    save_dir         = DISTILL_SAVE_DIR,
)

print(f'\nDistillation complete in {(time.time()-_t0)/60:.1f} min.')
print(f'Final distillation loss: {loss_history[-1]:.6f}')
print(f'Params saved to: {DISTILL_SAVE_DIR}/distilled_A_params.pkl')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(loss_history, color='tab:blue', lw=1.5)
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE Loss')
ax.set_title('Distillation Loss (A imitating B)')
ax.set_yscale('log')
plt.tight_layout()
plt.savefig(os.path.join(DISTILL_SAVE_DIR, 'distill_loss.pdf'), bbox_inches='tight')
plt.show()
print(f'Loss range: [{min(loss_history):.6f}, {max(loss_history):.6f}]')

---
## Stage 2 — Direct Evaluation of A_distilled (No Fine-Tuning)

Evaluate A_distilled *as a mechanism* — no IC fine-tuning.

Since A_distilled was trained by supervised imitation (not IC-constrained training),
it is unlikely to satisfy IC immediately. This step measures:
- **How faithfully** A can reproduce B's function (via revenue, IC regret).
- **Whether IC regret is small enough** to skip fine-tuning entirely.

The oracle policy is re-optimised from scratch with `REOPT_RESTARTS` independent
restarts to get a conservative (upper-bound) IC regret estimate.

In [ ]:
print('Stage 2: Re-optimising oracle against A_distilled ...')
_t0 = time.time()

distill_regret, distill_policy_params = reopt_cumulative_ic(
    auction             = auction,
    mechanism_network   = mechanism_net_A,
    mechanism_params    = distilled_params,
    policy_network      = policy_network,
    num_reopt_steps     = REOPT_STEPS,
    num_reopt_restarts  = REOPT_RESTARTS,
    lr_reopt            = REOPT_LR,
    num_samples         = REOPT_SAMPLES,
    seed                = SEED + 1000,
)

print(f'IC re-opt complete in {(time.time()-_t0)/60:.1f} min.')
print(f'A_distilled IC regret (reopt upper bound): {distill_regret:.4f}')
print(f'IC threshold: 0.05  →  {"PASS" if distill_regret <= 0.05 else "fail (fine-tuning needed)"}')

In [ ]:
_eval_keys = jax.random.split(jax.random.PRNGKey(55555), 32)

_ev_distill = eval_budget_mechanism_multikey(
    auction           = auction,
    mechanism_network = mechanism_net_A,
    mechanism_params  = distilled_params,
    profit_goal       = B_REVENUE_EVAL,  # use B's revenue as reference gap
    num_samples       = 256,
    keys              = _eval_keys,
    policy_network    = policy_network,
    policy_params     = distill_policy_params,
)

print_eval_report(_ev_distill, prefix='A_distilled (no fine-tune): ')
print(f'  IC regret (reopt):  {distill_regret:.4f}')
print(f'  Reference (B eval): {B_REVENUE_EVAL:.4f}')
print(f'  Revenue gap A_distilled vs B: {B_REVENUE_EVAL - _ev_distill["profit_stats"]["mean"]:.4f}')

# Save Stage 2 metrics
_stage2_path = os.path.join(DISTILL_SAVE_DIR, 'stage2_eval.json')
_stage2 = {
    'ic_regret_reopt':    float(distill_regret),
    'ic_pass':            bool(distill_regret <= 0.05),
    'revenue_eval':       float(_ev_distill['profit_stats']['mean']),
    'revenue_std':        float(_ev_distill['profit_stats']['std']),
    'buyer_surplus_eval': float(_ev_distill['buyer_surplus_stats']['mean']),
    'budget_viol':        float(_ev_distill['budget_viol_stats']['mean']),
    'ir_viol':            float(_ev_distill['ir_viol_stats']['mean']),
    'B_revenue_reference': float(B_REVENUE_EVAL),
}
with open(_stage2_path, 'w') as _f:
    json.dump(_stage2, _f, indent=2)
print(f'\nStage 2 metrics saved to: {_stage2_path}')

---
## Stage 3 — Warm-Start IC Fine-Tuning

Run the full binary search over `profit_goal` starting from `distilled_params` instead
of random init. Each trial of the binary search calls `nested_GD_budget` with
`init_mechanism_params=distilled_params`.

**If Stage 2 already passed IC (regret ≤ 0.05):** fine-tuning may not be needed; you
can skip directly to Stage 4 comparison. But running fine-tuning anyway can recover
more revenue.

**Expected runtime:** same as a normal Exp 1 binary search (~6–12h on GPU).

The binary search is fault-tolerant — safe to interrupt and resume.

In [ ]:
# Wrap run_binary_search to inject distilled_params as warm start.
# We patch train_with_checkpoints locally so binary_search calls nested_GD_budget
# with init_mechanism_params=distilled_params on every trial.

import budget_auction.train.binary_search as _bs_mod
_orig_train_fn = _bs_mod.train_with_checkpoints

def _warm_start_train(auction, mechanism_network, policy_network,
                      profit_goal, cfg, seed=42, verbose=True):
    """Drop-in replacement: passes distilled_params as warm start."""
    from budget_auction.train.run_acceptance import nested_GD_budget
    alpha = cfg['alpha_base'] * (profit_goal / cfg['pg_base'])
    if verbose:
        print(f'  [warm-start train] profit_goal={profit_goal:.3f}  alpha={alpha:.1e}  '
              f'(starting from A_distilled params)')
    logs = nested_GD_budget(
        auction             = auction,
        mechanism_network   = mechanism_network,
        policy_network      = policy_network,
        profit_goal         = profit_goal,
        num_outer_epochs    = cfg['num_outer_epochs'],
        lr_theta            = cfg['lr_theta'],
        num_inner_steps     = cfg['num_inner_steps'],
        lr_w                = cfg['lr_w'],
        num_samples_train   = cfg['num_samples_train'],
        alpha_weight        = alpha,
        beta_weight         = cfg['beta_weight'],
        print_iter          = cfg.get('print_iter', 100),
        checkpoint_thresh   = cfg['checkpoint_thresh'],
        checkpoint_interval = cfg['checkpoint_interval'],
        init_mechanism_params = distilled_params,   # ← warm start
        seed                = seed,
    )
    checkpoints = logs['checkpoint_mech']
    final_reg = logs['reg'][-1] if logs['reg'] else float('nan')
    final_ir  = logs['ir'][-1]  if logs['ir']  else float('nan')
    final_gap = logs['gap'][-1] if logs['gap'] else float('nan')
    checkpoints_with_final = checkpoints + [{
        'epoch': cfg['num_outer_epochs'] - 1,
        'reg_train': final_reg, 'ir_train': final_ir, 'gap_train': final_gap,
        'mechanism_params': logs['mechanism_params'],
        'policy_params':    logs['policy_params'],
    }]
    if verbose:
        print(f'  [warm-start train] done — {len(checkpoints)} threshold-passing checkpoints '
              f'(+ final); total candidates: {len(checkpoints_with_final)}')
    return checkpoints_with_final, logs

# Monkey-patch so run_binary_search uses the warm-start version
_bs_mod.train_with_checkpoints = _warm_start_train
print('Warm-start patch applied. run_binary_search will now use A_distilled params.')

In [ ]:
warmstart_summary = run_binary_search(
    auction           = auction,
    mechanism_network = mechanism_net_A,
    policy_network    = policy_network,
    variant           = 'A_distilled',
    save_dir          = WARMSTART_SAVE_DIR,
    cfg               = WARMSTART_BS_CFG,
    base_seed         = SEED,
    verbose           = True,
    time_budget_hours = TIME_BUDGET_HOURS,
    force_restart     = False,
)

# Restore original train function
_bs_mod.train_with_checkpoints = _orig_train_fn
print('Warm-start patch removed.')

print('\n' + '='*60)
print('Warm-start binary search complete:')
print(f'  pg_lo (highest feasible): {warmstart_summary["pg_lo"]:.3f}')
print(f'  pg_hi (infeasible):       {warmstart_summary["pg_hi"]:.3f}')
print(f'  Converged: {warmstart_summary["converged"]}')
if warmstart_summary['best_mechanism']:
    _bm = warmstart_summary['best_mechanism']
    print(f'  Best mechanism: rev={_bm["revenue_eval"]:.4f}  '
          f'IC={_bm["regret_eval"]:.4f}  surplus={_bm.get("buyer_surplus_eval", float("nan")):.4f}')

---
## Stage 4 — Final Comparison

Load results from all runs and build the comparison table:

| Variant | Revenue | IC Regret | Buyer Surplus | Notes |
|---|---|---|---|---|
| A (cold-start) | from binary search | | | Baseline |
| A_distilled (no fine-tune) | Stage 2 | | | Direct imit. |
| A_distilled + warm-start | Stage 3 | | | Warm-start fine-tune |
| B | reference | | | Target |

In [ ]:
# Load Variant A cold-start result from binary search
_a_summary_path = 'outputs/public_budget/binary_search/variant_A/bs_summary.json'
if os.path.exists(_a_summary_path):
    with open(_a_summary_path) as _f:
        _a_sum = json.load(_f)
    _a_bm = _a_sum.get('best_mechanism', {})
    A_REVENUE_EVAL  = _a_bm.get('revenue_eval', float('nan'))
    A_REGRET_EVAL   = _a_bm.get('regret_eval',  float('nan'))
    A_SURPLUS_EVAL  = _a_bm.get('buyer_surplus_eval', float('nan'))
    print(f'Loaded A cold-start: rev={A_REVENUE_EVAL:.4f}  IC={A_REGRET_EVAL:.4f}')
else:
    print(f'A cold-start summary not found at {_a_summary_path} — set manually.')
    A_REVENUE_EVAL = float('nan')
    A_REGRET_EVAL  = float('nan')
    A_SURPLUS_EVAL = float('nan')

In [ ]:
# Load warm-start fine-tuned best mechanism for final evaluation
_ws_bm_info = warmstart_summary.get('best_mechanism')
_ws_bm_info = _ws_bm_info if _ws_bm_info else None

# Fall back to trial records if summary is stale
if _ws_bm_info is None:
    _ws_results_path = os.path.join(WARMSTART_SAVE_DIR, 'bs_results.json')
    if os.path.exists(_ws_results_path):
        with open(_ws_results_path) as _f:
            _ws_trials = json.load(_f)
        _feas = [t for t in _ws_trials if t.get('any_feasible') and t.get('best_mech_pkl')]
        _feas.sort(key=lambda t: t.get('best_revenue_eval') or 0.0, reverse=True)
        if _feas:
            _ws_bm_info = _feas[0]

if _ws_bm_info is not None:
    _ws_pkl = _ws_bm_info.get('mech_pkl') or _ws_bm_info.get('best_mech_pkl')
    with open(_ws_pkl, 'rb') as _f:
        _ws_best = pickle.load(_f)

    # Final eval of warm-start best
    _eval_keys_final = jax.random.split(jax.random.PRNGKey(77777), 64)
    _ws_reopt_reg, _ws_policy_params = reopt_cumulative_ic(
        auction=auction, mechanism_network=mechanism_net_A,
        mechanism_params=_ws_best['mechanism_params'],
        policy_network=policy_network,
        num_reopt_steps=REOPT_STEPS, num_reopt_restarts=REOPT_RESTARTS,
        lr_reopt=REOPT_LR, num_samples=REOPT_SAMPLES, seed=SEED + 9999,
    )
    _ws_ev = eval_budget_mechanism_multikey(
        auction=auction, mechanism_network=mechanism_net_A,
        mechanism_params=_ws_best['mechanism_params'],
        profit_goal=_ws_best['profit_goal'],
        num_samples=512, keys=_eval_keys_final,
        policy_network=policy_network, policy_params=_ws_policy_params,
    )
    WS_REVENUE_EVAL  = float(_ws_ev['profit_stats']['mean'])
    WS_REGRET_EVAL   = float(_ws_reopt_reg)
    WS_SURPLUS_EVAL  = float(_ws_ev['buyer_surplus_stats']['mean'])
    print_eval_report(_ws_ev, prefix='A_distilled + warm-start: ')
    print(f'  IC regret (reopt): {_ws_reopt_reg:.4f}')
else:
    print('No feasible warm-start mechanism found — Stage 3 may need more trials.')
    WS_REVENUE_EVAL = float('nan')
    WS_REGRET_EVAL  = float('nan')
    WS_SURPLUS_EVAL = float('nan')

In [ ]:
# Load Stage 2 metrics
with open(os.path.join(DISTILL_SAVE_DIR, 'stage2_eval.json')) as _f:
    _s2 = json.load(_f)

# ── Print comparison table ─────────────────────────────────────────────────
_header = f"{'Variant':<30} | {'Revenue':>10} | {'IC Regret':>10} | {'Surplus':>9} | {'IC pass?':>8}"
_sep    = '-' * len(_header)
print(_header)
print(_sep)

_rows = [
    ('A (cold-start)',             A_REVENUE_EVAL,                A_REGRET_EVAL,                A_SURPLUS_EVAL),
    ('A_distilled (no fine-tune)', _s2['revenue_eval'],           _s2['ic_regret_reopt'],       _s2['buyer_surplus_eval']),
    ('A_distilled + warm-start',   WS_REVENUE_EVAL,               WS_REGRET_EVAL,               WS_SURPLUS_EVAL),
    ('B (target)',                  B_REVENUE_EVAL,                float('nan'),                 float('nan')),
]

for _name, _rev, _ic, _sur in _rows:
    _ic_pass = 'PASS' if _ic <= 0.05 else ('fail' if not np.isnan(_ic) else 'n/a')
    print(f"{_name:<30} | {_rev:>10.4f} | {_ic:>10.4f} | {_sur:>9.4f} | {_ic_pass:>8}")

print()
print('Interpretation:')
_d_gap = B_REVENUE_EVAL - _s2['revenue_eval']
_ws_gap = B_REVENUE_EVAL - WS_REVENUE_EVAL
_a_gap  = B_REVENUE_EVAL - A_REVENUE_EVAL
print(f'  A/B revenue gap (cold-start):             {_a_gap:.4f}')
print(f'  A_distilled/B revenue gap (direct):       {_d_gap:.4f}')
print(f'  A_distilled+finetune/B revenue gap:       {_ws_gap:.4f}')
if not np.isnan(WS_REVENUE_EVAL) and WS_REGRET_EVAL <= 0.05:
    _closed = (_a_gap - _ws_gap) / _a_gap * 100 if _a_gap > 0 else float('nan')
    print(f'  Fraction of A/B gap closed by warm-start: {_closed:.1f}%')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

_labels = ['A\n(cold-start)', 'A_distilled\n(no fine-tune)', 'A_distilled\n+warm-start', 'B\n(target)']
_revenues = [A_REVENUE_EVAL, _s2['revenue_eval'], WS_REVENUE_EVAL, B_REVENUE_EVAL]
_colors   = ['tab:blue', 'tab:orange', 'tab:green', 'tab:red']
_ic_pass  = [True, _s2['ic_pass'], WS_REGRET_EVAL <= 0.05, True]

_bars = ax.bar(_labels, _revenues, color=_colors, alpha=0.8, edgecolor='k', linewidth=0.8)
for _bar, _pass in zip(_bars, _ic_pass):
    _x = _bar.get_x() + _bar.get_width() / 2
    _y = _bar.get_height() + 0.005
    ax.text(_x, _y, '✓' if _pass else '✗', ha='center', va='bottom',
            fontsize=14, color='green' if _pass else 'red')

ax.set_ylabel('Revenue (eval)')
ax.set_title('A_distilled Experiment — Exp 1 Revenue Comparison')
ax.set_ylim(0, max(r for r in _revenues if not np.isnan(r)) * 1.15)
ax.axhline(B_REVENUE_EVAL, color='tab:red', linestyle='--', lw=1.0, alpha=0.6, label='B target')
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(DISTILL_SAVE_DIR, 'distill_comparison.pdf'), bbox_inches='tight')
plt.show()

In [ ]:
_final_summary = {
    'experiment':   'A_distilled (Exp 1 — Public-State Budget)',
    'distillation': {
        'num_trajectories': NUM_TRAJECTORIES,
        'num_epochs':       DISTILL_EPOCHS,
        'final_loss':       float(loss_history[-1]),
        'B_pkl_source':     _b_pkl,
    },
    'stage2_direct_eval': _s2,
    'stage3_warmstart': {
        'pg_lo':          warmstart_summary.get('pg_lo'),
        'pg_hi':          warmstart_summary.get('pg_hi'),
        'converged':      warmstart_summary.get('converged'),
        'revenue_eval':   WS_REVENUE_EVAL,
        'ic_regret_reopt': WS_REGRET_EVAL,
        'buyer_surplus':   WS_SURPLUS_EVAL,
    },
    'references': {
        'A_coldstart_revenue': A_REVENUE_EVAL,
        'B_revenue':           B_REVENUE_EVAL,
    },
}
_summary_out = os.path.join(DISTILL_SAVE_DIR, 'distill_experiment_summary.json')
with open(_summary_out, 'w') as _f:
    json.dump(_final_summary, _f, indent=2)
print(f'Final summary saved to: {_summary_out}')